In [7]:
!ls

AIMD.ipynb VDATCAR    XDATCAR


In [8]:
import numpy as np
import random

In [9]:
coord = []  # This list will hold a tuple (of 45 coordinate tuples) for each block.
lattice_vectors_str = """
 8.364366    0.000000    0.000000
 4.182183    7.243754    0.000000
 0.000000    0.000000   29.105968
"""
lattice_vectors_lines = lattice_vectors_str.strip().split('\n')
lattice_vectors_list = [list(map(float, line.split())) for line in lattice_vectors_lines]
lattice_vectors = np.array(lattice_vectors_list)
with open("XDATCAR", "r") as f:
    for line in f:
        # Look for the header that starts a new configuration block
        if line.strip().startswith("Direct configuration="):
            block = []
            for _ in range(45):
                coordinate_line = next(f).strip()
                coords = tuple(map(float, coordinate_line.split()))
                block.append(coords)
            block = np.dot(block, lattice_vectors)
            coord.append(np.array(block))
print(len(coord))
print(coord[0])

499
[[8.36436600e+00 1.60972310e+00 9.99999971e+00]
 [2.78812197e+00 1.60972310e+00 9.99999971e+00]
 [5.57624403e+00 1.60972310e+00 9.99999971e+00]
 [9.75842703e+00 4.02430781e+00 9.99999971e+00]
 [4.18218300e+00 4.02430781e+00 9.99999971e+00]
 [6.97030506e+00 4.02430781e+00 9.99999971e+00]
 [1.11524880e+01 6.43889245e+00 9.99999971e+00]
 [5.57624399e+00 6.43889245e+00 9.99999971e+00]
 [8.36436604e+00 6.43889245e+00 9.99999971e+00]
 [0.00000000e+00 0.00000000e+00 1.22764919e+01]
 [2.78812197e+00 0.00000000e+00 1.22764919e+01]
 [5.57624403e+00 0.00000000e+00 1.22764919e+01]
 [1.39406099e+00 2.41458464e+00 1.22764919e+01]
 [4.18218296e+00 2.41458464e+00 1.22764919e+01]
 [6.97030501e+00 2.41458464e+00 1.22764919e+01]
 [2.78812201e+00 4.82916936e+00 1.22764919e+01]
 [5.57624399e+00 4.82916936e+00 1.22764919e+01]
 [8.36436604e+00 4.82916936e+00 1.22764919e+01]
 [1.39534872e+00 8.04507690e-01 1.45526365e+01]
 [4.18263702e+00 8.05420693e-01 1.45554982e+01]
 [6.97005224e+00 8.04227719e-01 1.45

In [10]:
def read_vdatcar_blocks(filepath):
    blocks = []
    current_block = []
    
    with open(filepath, 'r') as f:
        for line in f:
            stripped = line.strip()
            # Skip KE/TEMP lines
            if stripped.startswith("KINETIC ENERGY") or stripped.startswith("TEMP EFF"):
                continue
            # Check if the line has exactly 3 float-like elements
            parts = stripped.split()
            if len(parts) == 3:
                try:
                    floats = list(map(float, parts))
                    current_block.append(floats)
                except ValueError:
                    continue  # skip non-numeric lines
                # If we've collected 45 lines, save the block
                if len(current_block) == 45:
                    blocks.append(np.array(current_block))
                    current_block = []
    return np.array(blocks)  # shape will be (num_blocks, 45, 3)

# Usage:
file_path = 'VDATCAR'  # Replace with your actual file name
vibrational_blocks = read_vdatcar_blocks(file_path)

print(len(vibrational_blocks))
print(vibrational_blocks[0])

499
[[ 8.36436600e+00  0.00000000e+00  0.00000000e+00]
 [ 4.18218300e+00  7.24375400e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  2.91059680e+01]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.000000

In [11]:
def N2_gen():
    a1 = np.array([5.5948640339220930, 0.0, 0.0])
    a2 = np.array([2.7974320169610465, 4.8452943840964133, 0.0])
    
    # Grid size
    n = 10
    height = 24  # z-position of molecule center
    
    # Generate 10x10 grid points inside the surface
    grid_points = []
    for i in range(n):
        for j in range(n):
            u = (i + 1) / (n + 1)
            v = (j + 1) / (n + 1)
            point = u * a1 + v * a2
            point[2] = height  # set z = 12.5
            grid_points.append(point)
    
    # Store N2 molecules as (100, 2, 3) array
    n2_molecules = np.zeros((len(grid_points), 2, 3))
    
    for idx, center in enumerate(grid_points):
        # Random spherical angles
        theta = np.random.uniform(0, np.pi)
        phi = np.random.uniform(0, 2 * np.pi)
    
        # Direction vector (unit)
        dx = np.sin(theta) * np.cos(phi)
        dy = np.sin(theta) * np.sin(phi)
        dz = np.cos(theta)
        direction = np.array([dx, dy, dz])
    
        # Offset from center
        bond_length = random.uniform(0.9, 1.3)
        offset = (bond_length / 2) * direction
    
        # N1 and N2 atom coordinates
        atom1 = center + offset
        atom2 = center - offset
    
        # Store in array
        n2_molecules[idx, 0] = atom1  # N1
        n2_molecules[idx, 1] = atom2  # N2
    return n2_molecules
    
n2_molecules = N2_gen()
print(n2_molecules)

[[[ 6.02447085e-01  4.55749502e-01  2.35025290e+01]
  [ 9.23424924e-01  4.25213113e-01  2.44974710e+01]]

 [[ 7.38736474e-01  8.94851337e-01  2.45425236e+01]
  [ 1.29575954e+00  8.67073893e-01  2.34574764e+01]]

 [[ 1.07701671e+00  1.03500047e+00  2.43708788e+01]
  [ 1.46610331e+00  1.60788737e+00  2.36291212e+01]]

 [[ 1.13064787e+00  1.61011804e+00  2.35521762e+01]
  [ 1.92109615e+00  1.91373243e+00  2.44478238e+01]]

 [[ 1.79986523e+00  2.19066086e+00  2.34441777e+01]
  [ 1.76050280e+00  2.21415222e+00  2.45558223e+01]]

 [[ 1.71557165e+00  2.60807412e+00  2.43438030e+01]
  [ 2.35342037e+00  2.67770157e+00  2.36561970e+01]]

 [[ 2.11718493e+00  3.21813755e+00  2.35939650e+01]
  [ 2.46043110e+00  2.94860076e+00  2.44060350e+01]]

 [[ 2.44801220e+00  3.82615406e+00  2.45643198e+01]
  [ 2.63822783e+00  3.22154686e+00  2.34356802e+01]]

 [[ 3.24274214e+00  3.68671781e+00  2.38136329e+01]
  [ 2.35212189e+00  4.24194573e+00  2.41863671e+01]]

 [[ 3.05004373e+00  4.40822477e+00  2.35155434

In [12]:
def write_vasp_style(data, data2, data3, vel, filename):
    header = """Pd N                                    
   1.00000000000000
     8.3643661146556720    0.0000000000000000    0.0000000000000000
     4.1821830573278360    7.2437535418455532    0.0000000000000000
     0.0000000000000000    0.0000000000000000   29.1059684456587782
   Pd  N
    45     2
Selective dynamics
Cartesian
"""

    def format_block_with_flags(data_block):
        lines = []
        for i, row in enumerate(data_block):
            coord = f"{row[0]: .12f}  {row[1]: .12f}  {row[2]: .12f}"
            flag = "  F  F  F" if i < 18 else "  T  T  T"
            lines.append(coord + flag)
        return lines
    
    def format_block_plain(data_block):
        return [
            f"{row[0]: .12f}  {row[1]: .12f}  {row[2]: .12f}"
            for row in data_block
        ]
    
    def format_data3_with_T_flags(data_block):
        return [
            f"{row[0]: .12f}  {row[1]: .12f}  {row[2]: .12f}  T  T  T"
            for row in data_block
        ]
    
    with open(filename, 'w') as f:
        # Write header
        f.write(header)
    
        # Write first block (data) with flags
        f.writelines(line + '\n' for line in format_block_with_flags(data))
    
        # Write data3 (N2 molecule) with 'T T T' flags
        f.writelines(line + '\n' for line in format_data3_with_T_flags(data3))
    
        # Blank line
        f.write('\n')
    
        # Write second block (data2) without flags
        f.writelines(line + '\n' for line in format_block_plain(data2))
    
        # Append the velocity vector
        vel_line = f"{0.0: .12f}  {0.0: .12f}  {-vel: .12f}"
        f.write(vel_line + '\n')
        f.write(vel_line + '\n')

In [17]:
for k in range(0,100):
    rand_int = random.randint(0, len(coord))
    write_vasp_style(coord[rand_int], vibrational_blocks[rand_int], n2_molecules[k], vel=0.074247632, filename=f"{k+1}.vasp")

In [16]:
for k in range(100):
    print(k)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
